# Era5

ERA5 via CDS (2015-2021) and DestinE Earth Data Hub zarr (2022-2025); window extraction and lag check.

Cells are the original session code in execution order (exported verbatim; only secrets handling and artifact paths were changed). Each cell starts with a comment giving its original index, conda environment and run status. Exploratory and failed cells are kept, so later cells that rely on their partial state still make sense.


In [ ]:
# Setup: run from the repository root; helper modules live in prod/
import os, sys
if os.path.basename(os.getcwd())=='notebooks': os.chdir('..')
sys.path.insert(0, 'prod')
os.makedirs('out', exist_ok=True)
from style import apply_figure_style  # stand-in for the figure style used in the session


In [144]:
%%bash
# [orig cell 143 | env: precip | status: ok]
mkdir -p prod/era5 && timeout 900 python - <<'EOF' 2>&1 | grep -v -i "key" | tail -15
import cdsapi
c=cdsapi.Client(quiet=False, progress=False)
r=c.retrieve('reanalysis-era5-single-levels',{'product_type':['reanalysis'],'variable':['total_precipitation'],'year':['2020'],'month':['05'],'day':['26'],
  'time':[f'{h:02d}:00' for h in range(24)],'area':[32.5,-89.5,23.0,-78.5],'data_format':'netcdf','download_format':'unarchived'})
r.download('prod/era5/test.nc'); print('ok')
EOF
ls -la prod/era5/

/bin/bash: timeout: command not found
total 0
drwxr-xr-x@  2 mehdihatamigoloujeh  staff    64 Oct 19 08:49 .
drwxr-xr-x@ 76 mehdihatamigoloujeh  staff  2432 Oct 19 08:49 ..


In [146]:
%%bash
# [orig cell 145 | env: precip | status: ok]
python - <<'EOF' 2>&1 | grep -v -iE "\bkey\b|token" | tail -15
import cdsapi
c=cdsapi.Client(quiet=False, progress=False, timeout=120)
r=c.retrieve('reanalysis-era5-single-levels',{'product_type':['reanalysis'],'variable':['total_precipitation'],'year':['2020'],'month':['05'],'day':['26'],
  'time':[f'{h:02d}:00' for h in range(24)],'area':[32.5,-89.5,23.0,-78.5],'data_format':'netcdf','download_format':'unarchived'})
r.download('prod/era5/test.nc'); print('ok')
EOF
ls -la prod/era5/

Recovering from connection error [HTTPSConnectionPool(host='object-store.os-api.cci2.ecmwf.int', port=443): Max retries exceeded with url: /cci2-prod-cache-3/2026-10-19/bbd7da70a02baa15f860db43faaf23.nc (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden (host object-store.os-api.cci2.ecmwf.int not on the allowlist)')))], attempt 4 of 500
Retrying in 120 seconds
Recovering from connection error [HTTPSConnectionPool(host='object-store.os-api.cci2.ecmwf.int', port=443): Max retries exceeded with url: /cci2-prod-cache-3/2026-10-19/bbd7da70a02baa15f860db43faaf23.nc (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden (host object-store.os-api.cci2.ecmwf.int not on the allowlist)')))], attempt 5 of 500
Retrying in 120 seconds
Recovering from connection error [HTTPSConnectionPool(host='object-store.os-api.cci2.ecmwf.int', port=443): Max retries exceeded with url: /cci2-prod-cache-3/2026-10-19/bbd

In [147]:
# [orig cell 146 | env: precip | status: ok]
t=xr.open_dataset('prod/era5/test.nc'); print(t); print((t.tp.sel(latitude=25.75,longitude=-80.25)*1000).values.round(1))

<xarray.Dataset> Size: 170kB
Dimensions:     (valid_time: 24, latitude: 39, longitude: 45)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 192B 2020-05-26 ... 2020-05-26T23...
    expver      (valid_time) <U4 384B ...
  * latitude    (latitude) float64 312B 32.5 32.25 32.0 ... 23.5 23.25 23.0
  * longitude   (longitude) float64 360B -89.5 -89.25 -89.0 ... -78.75 -78.5
    number      int64 8B ...
Data variables:
    tp          (valid_time, latitude, longitude) float32 168kB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-10-19T12:50 GRIB to CDM+CF via cfgrib-0.9.1...
[1.  2.5 2.1 1.8 0.3 0.3 0.3 1.3 3.9 6.3 4.9 2.5 0.4 0.7 2.7 2.6 5.7 1.8
 0.8 1.8 0.8 0.3 4.  3.5]


In [148]:
# [orig cell 147 | env: precip | status: ok]
def era5_windows(ds):
    fs_=sorted(glob.glob(f'prod/era5/{ds}_20*.nc')); print(ds,len(fs_),'files')
    X=xr.open_mfdataset(fs_,combine='by_coords') if len(fs_)>1 else xr.open_dataset(fs_[0])
    tp=X.tp.load()*1000.0   # m -> mm
    vt=pd.DatetimeIndex(tp.valid_time.values)
    if ds=='era5land':
        acc=tp.values; hr=np.empty_like(acc); hr[0]=acc[0]
        hr[1:]=acc[1:]-acc[:-1]
        first=(vt.hour==1); hr[first]=acc[first]     # 01 UTC value is the 00–01 accumulation
        hr[hr<0]=0; vals=hr
    else: vals=tp.values
    start=vt-pd.Timedelta(hours=1)    # stamp = hour end -> hour start
    la=tp.latitude.values; lo=tp.longitude.values; la2,lo2=np.meshgrid(la,lo,indexing='ij')
    W_=cm.window_index(la2,lo2); pos=pd.Series(np.arange(len(start)),index=start)
    out=np.full((len(Hneed),len(Vst),8,8),np.nan,np.float32)
    p=pos.reindex(Hneed).values; ok=~np.isnan(p)
    for k in np.where(ok)[0]: out[k]=cm.cut(vals[int(p[k])],W_)
    cm.save(ds,out,W_,Hneed); print(ds,'hours with data',int(np.isfinite(out[:,:,3,3]).any(1).sum()),'stations ok',sum(w['ok'] for w in W_))
print('ready')

ready


In [149]:
%%bash
# [orig cell 148 | env: precip | status: ok]
grep -v -iE "\bkey\b" prod/era5/dl_log.txt | tail -12; ls -la prod/era5 | tail -25

total 32952
drwxr-xr-x@  5 mehdihatamigoloujeh  staff       160 Oct 19 09:23 .
drwxr-xr-x@ 76 mehdihatamigoloujeh  staff      2432 Oct 19 08:49 ..
-rw-r--r--@  1 mehdihatamigoloujeh  staff         0 Oct 19 09:10 dl_log.txt
-rw-r--r--@  1 mehdihatamigoloujeh  staff  16257672 Oct 19 09:23 era5_2017.nc
-rw-r--r--@  1 mehdihatamigoloujeh  staff     91823 Oct 19 09:10 test.nc


In [151]:
# [orig cell 150 | env: precip | status: ok]
_k=os.environ['DESTINE_EDH_TOKEN']
base='https://data.earthdatahub.destine.eu/era5/'
for nm in ['reanalysis-era5-single-levels-v0.zarr','reanalysis-era5-land-no-antartica-v0.zarr','reanalysis-era5-land-v0.zarr']:
    for meta in ['.zmetadata','zarr.json']:
        r=requests.get(base+nm+'/'+meta,auth=('edh',_k),timeout=60)
        print(nm,meta,r.status_code,len(r.content))
        if r.ok: break

reanalysis-era5-single-levels-v0.zarr .zmetadata 200 307219
reanalysis-era5-land-no-antartica-v0.zarr .zmetadata 200 119121
reanalysis-era5-land-v0.zarr .zmetadata 404 146
reanalysis-era5-land-v0.zarr zarr.json 404 146


In [152]:
# [orig cell 151 | env: precip | status: ok]
import fsspec
for nm in ['reanalysis-era5-single-levels-v0.zarr','reanalysis-era5-land-no-antartica-v0.zarr']:
    m=requests.get(base+nm+'/.zmetadata',auth=('edh',_k),timeout=60).json()['metadata']
    for v in ['tp','time','valid_time','latitude','longitude']:
        a=m.get(f'{v}/.zarray'); at=m.get(f'{v}/.zattrs',{})
        if a: print(nm[:30],v,a['shape'],a['chunks'],a.get('compressor',{}).get('id') if a.get('compressor') else None,{k:at[k] for k in at if k in('units','long_name','GRIB_stepType','_ARRAY_DIMENSIONS','calendar')})

reanalysis-era5-single-levels- tp [757512, 721, 1440] [4320, 64, 64] blosc {'GRIB_stepType': 'accum', '_ARRAY_DIMENSIONS': ['valid_time', 'latitude', 'longitude'], 'long_name': 'Total precipitation', 'units': 'm'}
reanalysis-era5-single-levels- valid_time [757512] [753144] blosc {'_ARRAY_DIMENSIONS': ['valid_time'], 'calendar': 'proleptic_gregorian', 'units': 'hours since 1940-01-01 00:00:00'}
reanalysis-era5-single-levels- latitude [721] [721] blosc {'_ARRAY_DIMENSIONS': ['latitude'], 'long_name': 'latitude', 'units': 'degrees_north'}
reanalysis-era5-single-levels- longitude [1440] [1440] blosc {'_ARRAY_DIMENSIONS': ['longitude'], 'long_name': 'longitude', 'units': 'degrees_east'}
reanalysis-era5-land-no-antart tp [669840, 1472, 3600] [2880, 64, 64] blosc {'GRIB_stepType': 'accum', '_ARRAY_DIMENSIONS': ['valid_time', 'latitude', 'longitude'], 'long_name': 'Total precipitation', 'units': 'm'}
reanalysis-era5-land-no-antart valid_time [669840] [665472] blosc {'_ARRAY_DIMENSIONS': ['vali

In [153]:
%%bash
# [orig cell 152 | env: precip | status: ok]
cat > prod/edh_dl.py <<'EOF'
import xarray as xr, numpy as np, pandas as pd, aiohttp, sys, time
key=os.environ['DESTINE_EDH_TOKEN']
so={'client_kwargs':{'auth':aiohttp.BasicAuth('edh',key)}}
base='https://data.earthdatahub.destine.eu/era5/'
for ds,nm in [('era5','reanalysis-era5-single-levels-v0.zarr'),('era5land','reanalysis-era5-land-no-antartica-v0.zarr')]:
    t0=time.time()
    X=xr.open_zarr(base+nm,storage_options=so,chunks={})
    lon=X.longitude.values; lonc=np.where(lon>180,lon-360,lon)
    sel=X.tp.isel(longitude=np.where((lonc>=-89.5)&(lonc<=-78.5))[0]).sel(latitude=slice(32.5,23.0)).sel(valid_time=slice('2015-01-01T00:00','2026-01-01T01:00'))
    sel=sel.assign_coords(longitude=np.where(sel.longitude.values>180,sel.longitude.values-360,sel.longitude.values))
    out=sel.load()
    out.to_dataset(name='tp').to_netcdf(f'prod/era5/{ds}_edh.nc')
    print(ds,out.shape,str(out.valid_time.values[0]),str(out.valid_time.values[-1]),round((time.time()-t0)/60,1),'min',flush=True)
EOF
python prod/edh_dl.py 2>&1 | grep -v -i "auth" | tail -6

  File "~/.claude-science/conda/envs/precip/lib/python3.11/site-packages/aiohttp/connector.py", line 1263, in _create_connection
    _, proto = await self._create_direct_connection(req, traces, timeout)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "~/.claude-science/conda/envs/precip/lib/python3.11/site-packages/aiohttp/connector.py", line 1598, in _create_direct_connection
    raise ClientConnectorDNSError(req.connection_key, exc) from exc
aiohttp.client_exceptions.ClientConnectorDNSError: Cannot connect to host data.earthdatahub.destine.eu:443 ssl:default [nodename nor servname provided, or not known]


In [154]:
%%bash
# [orig cell 153 | env: precip | status: ok]
sed -i.bak "s/so={'client_kwargs':{'auth':aiohttp.BasicAuth('edh',key)}}/so={'client_kwargs':{'auth':aiohttp.BasicAuth('edh',key),'trust_env':True}}/" prod/edh_dl.py && grep -c trust_env prod/edh_dl.py && rm prod/edh_dl.py.bak && python prod/edh_dl.py 2>&1 | grep -v -i "auth" | tail -6

1
    await self._wait("readany")
  File "~/.claude-science/conda/envs/precip/lib/python3.11/site-packages/aiohttp/streams.py", line 364, in _wait
    with self._timer:
  File "~/.claude-science/conda/envs/precip/lib/python3.11/site-packages/aiohttp/helpers.py", line 759, in __exit__
    raise asyncio.TimeoutError from exc_val
TimeoutError


In [155]:
# [orig cell 154 | env: precip | status: ok]
import numcodecs, time, aiohttp
so={'client_kwargs':{'auth':aiohttp.BasicAuth('edh',_k),'trust_env':True}}
info={}
for ds,nm in [('era5','reanalysis-era5-single-levels-v0.zarr'),('era5land','reanalysis-era5-land-no-antartica-v0.zarr')]:
    X=xr.open_zarr(base+nm,storage_options=so,chunks={})
    vt=pd.DatetimeIndex(X.valid_time.values); la=X.latitude.values; lo=X.longitude.values
    m=requests.get(base+nm+'/.zmetadata',auth=('edh',_k),timeout=60).json()['metadata']; za=m['tp/.zarray']
    info[ds]=dict(nm=nm,vt=vt,la=la,lo=lo,za=za)
    print(ds,vt[0],vt[-1],la[[0,-1]],lo[[0,-1]],za['dimension_separator'] if 'dimension_separator' in za else '.',za['dtype'],za.get('fill_value'))
ds='era5'; I=info[ds]; ct,cy,cx=I['za']['chunks']
t_i=I['vt'].get_loc(pd.Timestamp('2020-05-26 00:00'))//ct
lonc=np.where(I['lo']>180,I['lo']-360,I['lo']); yi=np.where((I['la']<=32.5)&(I['la']>=23))[0]; xi=np.where((lonc>=-89.5)&(lonc<=-78.5))[0]
print('lat chunks',np.unique(yi//cy),'lon chunks',np.unique(xi//cx))
sep=I['za'].get('dimension_separator','.')
key=f"{t_i}{sep}{yi[0]//cy}{sep}{xi[0]//cx}"
t0=time.time(); r=requests.get(base+I['nm']+'/tp/'+key,auth=('edh',_k),timeout=600); print(r.status_code,len(r.content)/1e6,'MB',round(time.time()-t0,1),'s')

era5 1940-01-01 00:00:00 2026-05-31 23:00:00 [ 90. -90.] [  0.   359.75] . <f4 NaN
era5land 1950-01-01 00:00:00 2026-05-31 23:00:00 [ 90.  -57.1] [  0.  359.9] . <f4 NaN
lat chunks [3 4] lon chunks [16 17]
200 13.464367 MB 20.2 s


> Original cell 155 ended with status **error** in the session; it is kept for the record.

In [156]:
%%bash
# [orig cell 155 | env: precip | status: error]
python - <<'EOF' > prod/era5/dl_log.txt 2>&1
import cdsapi, os, concurrent.futures as cf, time
def get(job):
    ds,y=job; fn=f'prod/era5/{ds}_{y}.nc'
    if os.path.exists(fn): return job,'skip'
    name={'era5':'reanalysis-era5-single-levels','era5land':'reanalysis-era5-land'}[ds]
    req={'variable':['total_precipitation'],'year':[str(y)],'month':[f'{m:02d}' for m in range(1,13)],'day':[f'{d:02d}' for d in range(1,32)],
         'time':[f'{h:02d}:00' for h in range(24)],'area':[32.5,-89.5,23.0,-78.5],'data_format':'netcdf','download_format':'unarchived'}
    if ds=='era5': req['product_type']=['reanalysis']
    for a in range(3):
        try:
            c=cdsapi.Client(quiet=True,progress=False,timeout=300)
            c.retrieve(name,req).download(fn+'.part'); os.replace(fn+'.part',fn); return job,'ok'
        except Exception as e: err=str(e)[:200]; time.sleep(30)
    return job,'fail '+err
jobs=[('era5',y) for y in range(2015,2026)]+[('era5land',y) for y in range(2015,2026)]
with cf.ThreadPoolExecutor(6) as ex:
    for job,s in ex.map(get,jobs): print(job,s,flush=True)
EOF
grep -v -iE "\bkey\b" prod/era5/dl_log.txt | tail -25; ls -la prod/era5

[no output: the command (/usr/bin/sandbox-exec -p '(version 1)
(deny default (with message "CMD64_dWxpbWl0IC1uIDEwMjQ7IGV4cG9ydCBVVl9DQUNIRV9ESVI9IiRQV0QvLmNhY2hlL3V2IgpleHBvcnQgVE1QREl) wrote ZERO bytes to stdout and stderr before the abort — it hung before producing anything]

In [158]:
# [orig cell 157 | env: precip | status: ok]
def edh_windows(ds, deaccum=None):
    z=np.load(f'prod/era5/{ds}_edh.npz'); tp=z['tp']*1000.0; vt=pd.DatetimeIndex(z['vt']); la=z['lat']; lo=z['lon']
    if deaccum is None:   # detect daily accumulation: values rarely decrease within a day except at 01 UTC
        d=np.nanmean(np.diff(tp,axis=0)<-1e-6,axis=(1,2)); hrs=vt[1:].hour
        dec_mid=np.nanmean(d[(hrs>1)]); dec_01=np.nanmean(d[hrs==1]); deaccum= dec_mid<0.02 and dec_01>0.2
        print(ds,'frac decreasing (other hrs, 01UTC):',round(dec_mid,3),round(dec_01,3),'-> deaccumulate' if deaccum else '-> already hourly')
    if deaccum:
        hr=np.empty_like(tp); hr[0]=tp[0]; hr[1:]=tp[1:]-tp[:-1]; f=(vt.hour==1); hr[f]=tp[f]; hr[hr<0]=0; tp=hr
    start=vt-pd.Timedelta(hours=1)
    la2,lo2=np.meshgrid(la,lo,indexing='ij'); W_=cm.window_index(la2,lo2); pos=pd.Series(np.arange(len(start)),index=start)
    out=np.full((len(Hneed),len(Vst),8,8),np.nan,np.float32); p=pos.reindex(Hneed).values; ok=~np.isnan(p)
    for k in np.where(ok)[0]: out[k]=cm.cut(tp[int(p[k])],W_)
    cm.save(ds,out,W_,Hneed); print(ds,'hours',int(np.isfinite(out[:,:,3,3]).any(1).sum()),'stations ok',sum(w['ok'] for w in W_))
print('ok')

ok


> Original cell 158 ended with status **error** in the session; it is kept for the record.

In [159]:
%%bash
# [orig cell 158 | env: precip | status: error]
for d in era5 era5land; do echo "$d: $(ls prod/era5/ck_$d 2>/dev/null | grep -v tmp | wc -l) chunks"; done; ls -la prod/era5/*.npz 2>/dev/null

era5:        0 chunks
era5land:        0 chunks


In [161]:
%%bash
# [orig cell 160 | env: precip | status: ok]
for d in era5 era5land; do echo "$d: $(ls prod/era5/ck_$d 2>/dev/null | grep -v tmp | wc -l) chunks"; done; ls -la prod/era5/ | grep -E "npz|ck_"

era5:       20 chunks
era5land:        0 chunks
drwxr-xr-x@ 22 mehdihatamigoloujeh  staff       704 Oct 19 11:24 ck_era5


> Original cell 161 ended with status **error** in the session; it is kept for the record.

In [162]:
%%bash
# [orig cell 161 | env: precip | status: error]
cat > prod/edh_chunks.py <<'EOF'
import numpy as np, pandas as pd, requests, numcodecs, json, os, time, concurrent.futures as cf, xarray as xr, aiohttp
key=os.environ['DESTINE_EDH_TOKEN']; base='https://data.earthdatahub.destine.eu/era5/'
so={'client_kwargs':{'auth':aiohttp.BasicAuth('edh',key),'trust_env':True}}
for ds,nm in [('era5','reanalysis-era5-single-levels-v0.zarr'),('era5land','reanalysis-era5-land-no-antartica-v0.zarr')]:
    out_fn=f'prod/era5/{ds}_edh.npz'
    if os.path.exists(out_fn): continue
    t0=time.time()
    X=xr.open_zarr(base+nm,storage_options=so,chunks={}); vt=pd.DatetimeIndex(X.valid_time.values); la=X.latitude.values; lo=X.longitude.values
    za=requests.get(base+nm+'/.zmetadata',auth=('edh',key),timeout=120).json()['metadata']['tp/.zarray']
    ct,cy,cx=za['chunks']; comp=numcodecs.get_codec(za['compressor']); T,NY,NX=za['shape']
    lonc=np.where(lo>180,lo-360,lo)
    ti=np.where((vt>=pd.Timestamp('2015-01-01 00:00'))&(vt<=pd.Timestamp('2026-01-01 01:00')))[0]
    yi=np.where((la<=32.5)&(la>=23.0))[0]; xi=np.where((lonc>=-89.5)&(lonc<=-78.5))[0]
    keys=[(a,b,c) for a in np.unique(ti//ct) for b in np.unique(yi//cy) for c in np.unique(xi//cx)]
    print(ds,'chunks',len(keys),flush=True)
    os.makedirs(f'prod/era5/ck_{ds}',exist_ok=True)
    def get(k):
        fn=f'prod/era5/ck_{ds}/{k[0]}.{k[1]}.{k[2]}.npy'
        if os.path.exists(fn): return k,'skip'
        for a in range(6):
            try:
                r=requests.get(base+nm+'/tp/'+'.'.join(map(str,k)),auth=('edh',key),timeout=(30,900))
                if r.ok:
                    arr=np.frombuffer(comp.decode(r.content),dtype=za['dtype'])
                    nt=min(ct,T-k[0]*ct); ny_=min(cy,NY-k[1]*cy); nx_=min(cx,NX-k[2]*cx)
                    arr=arr.reshape(ct,cy,cx)[:nt,:ny_,:nx_] if arr.size==ct*cy*cx else arr.reshape(nt,ny_,nx_)
                    np.save(fn+'.tmp.npy',arr); os.replace(fn+'.tmp.npy',fn); return k,'ok'
                if r.status_code==404: return k,'404'
            except Exception as e: pass
            time.sleep(10*(a+1))
        return k,'fail'
    with cf.ThreadPoolExecutor(6) as ex:
        res=list(ex.map(get,keys))
    print(ds,pd.Series([s for _,s in res]).value_counts().to_dict(),round((time.time()-t0)/60,1),'min',flush=True)
    # assemble subset
    sub=np.full((len(ti),len(yi),len(xi)),np.nan,np.float32)
    for (a,b,c),s in res:
        fn=f'prod/era5/ck_{ds}/{a}.{b}.{c}.npy'
        if not os.path.exists(fn): continue
        arr=np.load(fn); tg=np.arange(a*ct,a*ct+arr.shape[0]); yg=np.arange(b*cy,b*cy+arr.shape[1]); xg=np.arange(c*cx,c*cx+arr.shape[2])
        tm=np.isin(ti,tg); ym=np.isin(yi,yg); xm=np.isin(xi,xg)
        sub[np.ix_(tm,ym,xm)]=arr[np.ix_(np.isin(tg,ti),np.isin(yg,yi),np.isin(xg,xi))]
    np.savez(out_fn,tp=sub,vt=vt[ti].values,lat=la[yi],lon=lonc[xi])
    print(ds,'assembled',sub.shape,'nan frac',round(float(np.isnan(sub).mean()),4),flush=True)
EOF
python prod/edh_chunks.py 2>&1 | grep -v -i auth | tail -8

[no output: the command (/usr/bin/sandbox-exec -p '(version 1)
(deny default (with message "CMD64_dWxpbWl0IC1uIDEwMjQ7IGV4cG9ydCBVVl9DQUNIRV9ESVI9IiRQV0QvLmNhY2hlL3V2IgpleHBvcnQgVE1QREl) wrote ZERO bytes to stdout and stderr before the abort — it hung before producing anything]

In [164]:
# [orig cell 163 | env: precip | status: ok]
I=info['era5']; ct,cy,cx=I['za']['chunks']
f=sorted(glob.glob('prod/era5/ck_era5/*.npy'))[0]; a,b,c=map(int,os.path.basename(f).split('.')[:3]); arr=np.load(f)
tg=I['vt'][a*ct:a*ct+arr.shape[0]]; yg=I['la'][b*cy:b*cy+arr.shape[1]]; xg=np.where(I['lo']>180,I['lo']-360,I['lo'])[c*cx:c*cx+arr.shape[2]]
cds=xr.open_mfdataset(sorted(glob.glob('prod/era5/era5_20*.nc')),combine='by_coords').tp
tt=tg[(tg>=pd.Timestamp('2015-01-01'))][:500]; yy=yg[(yg<=32.5)&(yg>=23)]; xx=xg[(xg>=-89.5)&(xg<=-78.5)]
e=arr[np.isin(tg,tt)][:, np.isin(yg,yy)][:,:,np.isin(xg,xx)]
cv=cds.sel(valid_time=tt,latitude=yy,longitude=xx).values
print(f, e.shape, cv.shape, 'max abs diff mm', float(np.nanmax(np.abs(e-cv))*1000), 'mean mm', float(np.nanmean(cv)*1000))

prod/era5/ck_era5/152.3.16.npy (500, 26, 6) (500, 26, 6) max abs diff mm 0.003814697265625 mean mm 0.1553642451763153


In [165]:
%%bash
# [orig cell 164 | env: precip | status: ok]
sed -e "s/for ds,nm in \[('era5','reanalysis-era5-single-levels-v0.zarr'),('era5land','reanalysis-era5-land-no-antartica-v0.zarr')\]:/for ds,nm in [('era5','reanalysis-era5-single-levels-v0.zarr')]:/" -e "s/pd.Timestamp('2015-01-01 00:00')/pd.Timestamp('2021-12-31 00:00')/" -e "s#out_fn=f'prod/era5/{ds}_edh.npz'#out_fn=f'prod/era5/{ds}_edh_2022_2025.npz'#" prod/edh_chunks.py > prod/edh_chunks_2225.py && grep -c "2021-12-31" prod/edh_chunks_2225.py && python prod/edh_chunks_2225.py 2>&1 | grep -v -i auth | tail -5

1
era5 chunks 36
era5 {'ok': 36} 17.8 min
era5 assembled (35090, 39, 45) nan frac 0.0


> Original cell 165 ended with status **error** in the session; it is kept for the record.

In [166]:
# [orig cell 165 | env: precip | status: error]
cds=xr.open_mfdataset(sorted(glob.glob('prod/era5/era5_20*.nc')),combine='by_coords').tp.load()
z=np.load('prod/era5/era5_edh_2022_2025.npz'); vt2=pd.DatetimeIndex(z['vt'])
assert np.allclose(cds.latitude.values,z['lat']) and np.allclose(cds.longitude.values,z['lon'])
vt1=pd.DatetimeIndex(cds.valid_time.values); keep2=vt2>vt1.max()
tp=np.concatenate([cds.values,z['tp'][keep2]]); vt=vt1.append(vt2[keep2])
print('merged',tp.shape,vt.min(),vt.max(),'gaps',int((np.diff(vt.values).astype('timedelta64[h]').astype(int)!=1).sum()),'CDS max',vt1.max())
np.savez('prod/era5/era5_edh.npz',tp=tp,vt=vt.values,lat=z['lat'],lon=z['lon'])
edh_windows('era5',deaccum=False)
NBe,me_=load('era5'); print('era5 lag',lagcorr_g(NBe))

merged (96434, 39, 45) 2015-01-01 00:00:00 2026-01-01 01:00:00 gaps 0 CDS max 2021-12-31 23:00:00
era5 hours 22744 stations ok 79


Traceback (most recent call last):
  File "<kernel:91>", line 9, in <module>
    NBe,me_=load('era5'); print('era5 lag',lagcorr_g(NBe))
             ^^^^^^^^^^^^
  File "<kernel:40>", line 13, in load
    masks=[common.neighborhood_masks(w) for w in meta['widx']]
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<kernel:40>", line 13, in <listcomp>
    masks=[common.neighborhood_masks(w) for w in meta['widx']]
           ^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'set' object has no attribute 'neighborhood_masks'


In [167]:
# [orig cell 166 | env: precip | status: ok]
common=cm
NBe,me_=load('era5'); print('era5 lag',lagcorr_g(NBe))

era5 lag {'metar': {-2: np.float64(0.151), -1: np.float64(0.182), 0: np.float64(0.2), 1: np.float64(0.194), 2: np.float64(0.165)}, 'onhour': {-2: np.float64(0.157), -1: np.float64(0.186), 0: np.float64(0.198), 1: np.float64(0.177), 2: np.float64(0.147)}}
